In [1]:
!pip install transformers datasets peft accelerate bitsandbytes
!pip install rouge-score nltk

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 16.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=d066d60cb0f45e6217cb77e8c12b87addba6ea4b4ecbdf5b7ccf5a7db95d7d0e
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge-score


In [2]:
import os
os.environ["TRANSFORMERS_VERBOSITY"] = "error"

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer, pipeline
from peft import LoraConfig, get_peft_model
import torch

In [3]:
# 1. LOAD DATASET (Assignment 1)
dataset = load_dataset("ccdv/arxiv-summarization", split="train[:3000]")

def format_data(example):
    return {
        "text": f"### Instruction: Summarize the following research:\n{example['article'][:300]}\n### Response: {example['abstract']}"
    }

dataset = dataset.map(format_data)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

section/train-00000-of-00015.parquet:   0%|          | 0.00/230M [00:00<?, ?B/s]

section/train-00001-of-00015.parquet:   0%|          | 0.00/228M [00:00<?, ?B/s]

section/train-00002-of-00015.parquet:   0%|          | 0.00/228M [00:00<?, ?B/s]

...

section/test-00000-of-00001.parquet:   0%|          | 0.00/105M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/203037 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/6436 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/6440 [00:00<?, ? examples/s]

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

In [4]:
# 2. LOAD BASE MODEL
from transformers import BitsAndBytesConfig
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    quantization_config=bnb_config
)

base_model.config.use_cache = False

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [5]:
# 3. APPLY LoRA (Fine-tuning)
from peft import prepare_model_for_kbit_training

base_model = prepare_model_for_kbit_training(base_model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none"
)

model = get_peft_model(base_model, lora_config)

In [6]:
# 4. TOKENIZATION
def tokenize(example):
    tokens = tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=256
    )
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

tokenized_dataset = dataset.map(tokenize)

tokenized_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

In [7]:
# 5. TRAINING
training_args = TrainingArguments(
    output_dir="./finetuned_model",
    per_device_train_batch_size=1,
    num_train_epochs=3,
    logging_steps=10,
    save_steps=50,
    fp16=True,
    remove_unused_columns=False
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset
)

trainer.train()


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


{'loss': '4.144', 'grad_norm': '1.955', 'learning_rate': '4.995e-05', 'epoch': '0.003333'}
{'loss': '3.208', 'grad_norm': '1.844', 'learning_rate': '4.989e-05', 'epoch': '0.006667'}
{'loss': '3.116', 'grad_norm': '2.409', 'learning_rate': '4.984e-05', 'epoch': '0.01'}
{'loss': '2.828', 'grad_norm': '3.863', 'learning_rate': '4.978e-05', 'epoch': '0.01333'}
{'loss': '3.161', 'grad_norm': '2.683', 'learning_rate': '4.973e-05', 'epoch': '0.01667'}


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


{'loss': '2.645', 'grad_norm': '2.15', 'learning_rate': '4.967e-05', 'epoch': '0.02'}
{'loss': '2.925', 'grad_norm': '2.457', 'learning_rate': '4.962e-05', 'epoch': '0.02333'}
{'loss': '3.099', 'grad_norm': 'nan', 'learning_rate': '4.956e-05', 'epoch': '0.02667'}
{'loss': '3.045', 'grad_norm': '13.73', 'learning_rate': '4.951e-05', 'epoch': '0.03'}
{'loss': '2.715', 'grad_norm': '2.877', 'learning_rate': '4.945e-05', 'epoch': '0.03333'}


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


...

/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


{'loss': '1.931', 'grad_norm': '2.883', 'learning_rate': '5.056e-07', 'epoch': '2.97'}
{'loss': '2.061', 'grad_norm': '3.098', 'learning_rate': '4.5e-07', 'epoch': '2.973'}
{'loss': '2.101', 'grad_norm': '3.061', 'learning_rate': '3.944e-07', 'epoch': '2.977'}
{'loss': '2.143', 'grad_norm': '3.075', 'learning_rate': '3.389e-07', 'epoch': '2.98'}
{'loss': '2.166', 'grad_norm': '3.212', 'learning_rate': '2.833e-07', 'epoch': '2.983'}


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


{'loss': '1.938', 'grad_norm': '3.255', 'learning_rate': '2.278e-07', 'epoch': '2.987'}
{'loss': '2.157', 'grad_norm': '3.441', 'learning_rate': '1.722e-07', 'epoch': '2.99'}
{'loss': '1.728', 'grad_norm': '3.974', 'learning_rate': '1.167e-07', 'epoch': '2.993'}
{'loss': '2.044', 'grad_norm': '2.347', 'learning_rate': '6.111e-08', 'epoch': '2.997'}
{'loss': '1.961', 'grad_norm': '2.814', 'learning_rate': '5.556e-09', 'epoch': '3'}
{'train_runtime': '3218', 'train_samples_per_second': '2.797', 'train_steps_per_second': '2.797', 'train_loss': '2.083', 'epoch': '3'}


TrainOutput(global_step=9000, training_loss=2.0830941734313964, metrics={'train_runtime': 3217.9626, 'train_samples_per_second': 2.797, 'train_steps_per_second': 2.797, 'train_loss': 2.0830941734313964, 'epoch': 3.0})

In [8]:
# 6. SAVE MODEL
model.save_pretrained("./finetuned_model")
tokenizer.save_pretrained("./finetuned_model")


('./finetuned_model/tokenizer_config.json',
 './finetuned_model/chat_template.jinja',
 './finetuned_model/tokenizer.json')

In [9]:
# 7. LOAD PIPELINES
base_pipe = pipeline(
    "text-generation",
    model=model_name,
    tokenizer=model_name
)

finetuned_pipe = pipeline(
    "text-generation",
    model="./finetuned_model",
    tokenizer="./finetuned_model"
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/88 [00:00<?, ?it/s]

In [10]:
# 8. COMPARISON
prompt = "### Instruction: Explain gradient descent\n### Response:"

base_output = base_pipe(prompt, max_new_tokens=100)[0]["generated_text"]
finetuned_output = finetuned_pipe(prompt, max_new_tokens=100)[0]["generated_text"]

print("\n===== BASE MODEL OUTPUT =====\n")
print(base_output)

print("\n===== FINE-TUNED MODEL OUTPUT =====\n")
print(finetuned_output)



===== BASE MODEL OUTPUT =====

### Instruction: Explain gradient descent
### Response: Gradient descent is a type of iterative algorithm that is used to optimize the parameters of a function using a sequence of small step changes. It works by iteratively updating the parameters in a direction that minimizes the error of the function. Gradient descent is a stochastic optimization algorithm, which means that it can be applied to non-linear and non-convex problems. The algorithm works by taking a small step in the direction of the gradient of the function, which is the direction that minim

===== FINE-TUNED MODEL OUTPUT =====

### Instruction: Explain gradient descent
### Response: we consider the l2 regularized logistic regression problem with a convex objective function defined as @xmath0. we derive the first and second order derivatives of the objective function @xmath1 and @xmath2, which are used to derive the gradient descent algorithm. we also derive the rate of convergence of the 

In [11]:
from rouge_score import rouge_scorer
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
import nltk

nltk.download('punkt')

# Reference (expected output from dataset)
example = dataset[0]
# Fixed prompt to match training data exactly
prompt = f"### Instruction: Summarize the following research:\n{example['article'][:300]}\n### Response:"
reference = example["abstract"]

# Forced greedy decoding (do_sample=False) for deterministic evaluation
base_output = base_pipe(prompt, max_new_tokens=150, do_sample=False, pad_token_id=tokenizer.eos_token_id)[0]["generated_text"]
finetuned_output = finetuned_pipe(prompt, max_new_tokens=150, do_sample=False, pad_token_id=tokenizer.eos_token_id)[0]["generated_text"]

# Extract generated responses
base_resp = base_output.split("### Response:")[-1].strip()
fine_resp = finetuned_output.split("### Response:")[-1].strip()

# ROUGE
scorer = rouge_scorer.RougeScorer(['rouge1', 'rougeL'], use_stemmer=True)

base_rouge = scorer.score(reference, base_resp)
fine_rouge = scorer.score(reference, fine_resp)

# BLEU
smooth = SmoothingFunction().method1

base_bleu = sentence_bleu([reference.split()], base_resp.split(), smoothing_function=smooth)
fine_bleu = sentence_bleu([reference.split()], fine_resp.split(), smoothing_function=smooth)

print("\n===== METRICS COMPARISON =====\n")

print("Base Model ROUGE-1:", base_rouge['rouge1'].fmeasure)
print("Fine-tuned Model ROUGE-1:", fine_rouge['rouge1'].fmeasure)

print("Base Model ROUGE-L:", base_rouge['rougeL'].fmeasure)
print("Fine-tuned Model ROUGE-L:", fine_rouge['rougeL'].fmeasure)

print("Base Model BLEU:", base_bleu)
print("Fine-tuned Model BLEU:", fine_bleu)

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.



===== METRICS COMPARISON =====

Base Model ROUGE-1: 0.35121951219512193
Fine-tuned Model ROUGE-1: 0.2672811059907834
Base Model ROUGE-L: 0.1951219512195122
Fine-tuned Model ROUGE-L: 0.17511520737327188
Base Model BLEU: 0.005184045304522133
Fine-tuned Model BLEU: 0.006210397517276085


In [ ]:
# 9. INTERACTIVE CHATBOT
print("\nModel ready! Type 'exit' to stop.\n")

while True:
    user_input = input("Enter your question: ")

    if user_input.lower() == "exit":
        break

    prompt = f"### Instruction: Explain in detail: {user_input}\n### Response:"

    output = finetuned_pipe(
        prompt,
        max_new_tokens=150,
        do_sample=True,
        temperature=0.7,
        top_p=0.9
    )

    answer = output[0]["generated_text"].split("### Response:")[-1].strip()

    print("\nAnswer:", answer)
    print("-" * 50)


Model ready! Type 'exit' to stop.

Enter your question: what is machine learning

Answer: we investigate the relationship between the frequency of occurrence of words and their frequency in a text by studying the topic of a given text. we use a topic modeling approach to analyze the topic distribution of words in a text. 
 we compare the results obtained by the two approaches with respect to the frequency of the top words and the frequency of the top topics. 
 we find that the two approaches can distinguish the topics with high accuracy, but they are not very effective for distinguishing the topics with low frequency.
--------------------------------------------------
Enter your question: transformers

Answer: we investigate the relationship between the frequency content and the time delay of the phase response of a two-port network with a transformer at the input and a rectifier at the output. 
 the frequency content of the phase response is estimated using the fomrulation of b. m. k